# 02 — Development windows and the naive benchmark

We use split-adjusted Close to forecast prices on a consistent retrospective split basis without dividend adjustment. Volume features remain deferred pending verification of adjustment semantics. See [data conventions](../docs/DATA_CONVENTIONS.md). The 2025 test prices never enter the constructed dataset or metrics. The source snapshot contains them, but the builder truncates before numerical processing.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.graph_objects as go
from market_forecasting.dataset import build_development_dataset, naive_predictions, error_metrics
root=Path.cwd()
if not (root/'configs/nvda.json').exists(): root=root.parent
config=json.loads((root/'configs/nvda.json').read_text())
payload=json.loads((root/'data/raw/NVDA_yahoo_snapshot.json').read_text())['chart']['result'][0]
dates=pd.to_datetime(payload['timestamp'],unit='s',utc=True).tz_convert('America/New_York').normalize().tz_localize(None)
close=pd.Series(payload['indicators']['quote'][0]['close'],index=dates)
dataset=build_development_dataset(close, config)
print('X shape (examples, sessions, features):',dataset.X.shape)
dataset.metadata.groupby('split').agg(examples=('target_date','size'),first_target=('target_date','min'),last_target=('target_date','max'))


## Boundary example
The first validation target follows the New Year holiday. Its context can include training observations because they were known before the target session. No model parameters are fitted using validation prices by this baseline.


In [ ]:
pred=naive_predictions(dataset)
mask=dataset.metadata['split'].eq('validation').to_numpy()
validation=dataset.metadata.loc[mask].copy()
validation['prediction']=pred[mask]
validation['actual']=dataset.y[mask]
validation.head()


## Validation metrics only
MAE measures the average absolute dollar error. RMSE weights large misses more heavily. Both are on the retrospective split-adjusted dollar basis. Subsequent model comparisons require the same target dates and price basis.


In [ ]:
error_metrics(dataset.y[mask],pred[mask])


In [ ]:
fig=go.Figure()
fig.add_scatter(x=validation.target_date,y=validation.actual,name='Actual close')
fig.add_scatter(x=validation.target_date,y=validation.prediction,name='Previous-session-close forecast')
fig.update_layout(title='Naive forecast — 2023–2024 validation only',xaxis_title='Target session',yaxis_title='Split-adjusted USD',template='plotly_white')
fig.show()


## Results interpretation
The forecast curve will closely track actual prices because it is the preceding session's close. Visual overlap is not proof of predictive skill. The baseline requires only the final Close in each window; the common 30-session representation supports comparison with subsequent LSTM models. No 2025 results have been computed.
